# 06 — Serving con MLflow Models

In [ ]:
!pip install mlflow lightgbm pandas numpy scikit-learn requests -q

In [ ]:
import os
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from mlflow.tracking import MlflowClient
import json
import warnings
warnings.filterwarnings('ignore')
# En el contenedor de Jupyter (make up) los runs van al servidor de MLflow
# (UI: http://localhost:5050). Fuera de Docker se guardan en ./mlruns.
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "mlruns"))

## 1. ¿Cómo funciona MLflow Models Serving?

```
Terminal: mlflow models serve --model-uri models:/nombre/Production --port 8001
                │
                ▼
         ┌──────────────┐
         │  Flask app   │  (creada automáticamente por MLflow)
         │  (interno)   │
         └──────┬───────┘
                │ mlflow.pyfunc.load_model()
                ▼
         ┌──────────────┐
         │  Tu modelo   │  (LightGBM, sklearn, cualquier pyfunc)
         └──────────────┘
                │
        POST /invocations
                │
         ┌──────────────┐
         │   Cliente    │
         └──────────────┘
```

MLflow Models Serving es la forma más rápida de servir un modelo registrado en MLflow.
Con un solo comando levanta una Flask app que expone el endpoint `/invocations`.

**Ventajas:**
- Cero código: 1 comando en terminal.
- Carga el modelo automáticamente desde el Registry.
- Acepta múltiples formatos de request (dataframe_records, dataframe_split, etc.).
- Valida inputs usando la `signature` del modelo.

**Limitaciones:**
- Single-threaded por defecto (no escala bien).
- Sin autenticación.
- No es production-grade sin infraestructura adicional.

## 2. Entrenar y registrar un modelo en MLflow

In [ ]:
rng = np.random.default_rng(42)
n = 1500
X = pd.DataFrame({
    "hour": rng.integers(0, 24, n),
    "temperature": rng.normal(26, 5, n),
    "is_peak": rng.integers(0, 2, n),
})
y = 1000 + 80*np.sin(2*np.pi*X["hour"]/24) + 3*X["temperature"] + rng.normal(0, 50, n)

mlflow.set_experiment("serving_demo")
with mlflow.start_run(run_name="model_for_serving") as run:
    model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
    model.fit(X, y)
    from mlflow.models.signature import infer_signature
    signature = infer_signature(X, model.predict(X))
    mlflow.lightgbm.log_model(model, "model", signature=signature, input_example=X.head(3))
    run_id = run.info.run_id

print(f"Modelo logueado. Run ID: {run_id}")
print(f"\nPara servir este modelo en un servidor real:")
print(f"  mlflow models serve --model-uri runs:/{run_id}/model --port 8001 --no-conda")

## 3. Formatos de request que acepta MLflow

In [ ]:
# Formato 1: dataframe_records (más común)
request_records = {
    "dataframe_records": [
        {"hour": 19, "temperature": 28.5, "is_peak": 1},
        {"hour": 8,  "temperature": 22.0, "is_peak": 0},
    ]
}
print("Formato dataframe_records:")
print(json.dumps(request_records, indent=2))

# Formato 2: dataframe_split
request_split = {
    "dataframe_split": {
        "columns": ["hour", "temperature", "is_peak"],
        "data": [[19, 28.5, 1], [8, 22.0, 0]]
    }
}
print("\nFormato dataframe_split:")
print(json.dumps(request_split, indent=2))

# Comando curl equivalente:
print("\nEquivalente con curl:")
print(f"""curl -X POST http://localhost:8001/invocations \\
  -H \"Content-Type: application/json\" \\
  -d '{json.dumps(request_records)}'""")

## 4. Cargar el modelo y predecir directamente (equivalente a POST /invocations)

In [ ]:
# En Colab no podemos levantar un servidor, pero podemos cargar el modelo directamente
model_uri = f"runs:/{run_id}/model"
loaded = mlflow.pyfunc.load_model(model_uri)

# Esto es exactamente lo que hace MLflow serving internamente
sample = pd.DataFrame({"hour": [19, 8, 14], "temperature": [28.5, 22.0, 30.0], "is_peak": [1, 0, 0]})
predictions = loaded.predict(sample)
print("Predicciones (equivalentes a POST /invocations):")
for i, (row, pred) in enumerate(zip(sample.itertuples(), predictions)):
    print(f"  Hora {row.hour}h, {row.temperature}\u00b0C \u2192 {pred:.1f} kWh")

## 5. Flask custom vs mlflow models serve

| Característica | Flask custom | mlflow models serve |
|------------------------|------------------|---------------------|
| Setup | Código propio | 1 comando |
| Autenticación | Implementar tú | No incluida |
| Batching | Implementar tú | Básico |
| Monitoreo | Implementar tú | No incluido |
| Versionado de modelos | Manual | Automático (Registry) |
| Escalabilidad | Tú decides | Single instance |
| Logging | Implementar tú | Básico |
| Validación de inputs | Pydantic/Manual | Via signature |
| Producción-grade | Con esfuerzo | NO |
| Cuándo usar | APIs personalizadas | Dev/Testing |

**Conclusión**: `mlflow models serve` es ideal para desarrollo y testing rápido.
Para producción real se necesita infraestructura adicional (SageMaker, Kubernetes, etc.).

## 6. Deploy en AWS SageMaker (simulación)

In [ ]:
# Simulación del flujo de SageMaker (dry_run=True)
print("=== Simulación: Deploy en AWS SageMaker ===")
print()
print("PASO 1 — Crear SageMaker Model:")
print('  sm_client.create_model(')
print('    ModelName="mlops-demand-forecaster",')
print('    PrimaryContainer={')
print('      "Image": "123456.dkr.ecr.us-east-1.amazonaws.com/mlflow-pyfunc:latest",')
print(f'      "ModelDataUrl": "s3://mlflow-artifacts/model",')
print('    },')
print('    ExecutionRoleArn="arn:aws:iam::123456:role/SageMakerRole",')
print('  )')
print()
print("PASO 2 — Crear EndpointConfig:")
print('  sm_client.create_endpoint_config(')
print('    EndpointConfigName="demand-forecaster-config",')
print('    ProductionVariants=[{')
print('      "InstanceType": "ml.t2.medium",')
print('      "InitialInstanceCount": 1,')
print('    }],')
print('  )')
print()
print("PASO 3 — Crear Endpoint (tarda ~5-10 minutos):")
print('  sm_client.create_endpoint(EndpointName="demand-forecaster-endpoint", ...)')
print()
print("Ventajas de SageMaker vs Flask:")
print("  - Autoscaling automático")
print("  - Alta disponibilidad (multi-AZ)")
print("  - Monitoring con CloudWatch incluido")
print("  - A/B testing con variants")

## Resumen

En este notebook aprendimos:

1. **MLflow serving**: con `mlflow models serve` se levanta automáticamente una Flask app que expone `/invocations`.
2. **Registro de modelos**: cómo entrenar un LightGBM, inferir la signature y loguearlo con MLflow.
3. **Formatos de request**: `dataframe_records` y `dataframe_split` son los más usados.
4. **Predicción directa**: `mlflow.pyfunc.load_model()` hace exactamente lo mismo que el servidor internamente.
5. **Comparativa**: Flask custom ofrece más control; `mlflow models serve` es más rápido para dev/testing.
6. **SageMaker**: para producción real, se necesita infraestructura managed (autoscaling, HA, monitoring).

**Próximo notebook**: `07_airflow_pipeline_walkthrough.ipynb` — orquestar el pipeline MLOps completo con Apache Airflow.